In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### 3.1 Basic Data Understanding

In [2]:
df=pd.read_csv("Algerian_forest_fires_dataset_UPDATE.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          122 non-null    int64  
 1   month        122 non-null    int64  
 2   year         122 non-null    int64  
 3   Temperature  122 non-null    int64  
 4    RH          122 non-null    int64  
 5    Ws          122 non-null    int64  
 6   Rain         122 non-null    float64
 7   FFMC         122 non-null    float64
 8   DMC          122 non-null    float64
 9   DC           122 non-null    object 
 10  ISI          122 non-null    float64
 11  BUI          122 non-null    float64
 12  FWI          122 non-null    object 
 13  Classes      121 non-null    object 
dtypes: float64(5), int64(6), object(3)
memory usage: 13.5+ KB


In [4]:
df.describe()

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,ISI,BUI
count,122.000000,122.000000,122.0,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000
mean,15.754098,7.500000,2012.0,33.163934,55.901639,15.008197,0.678689,81.102459,17.031967,5.892623,17.903279
std,8.843274,1.115259,0.0,3.675608,15.716186,2.692186,1.486759,12.244064,12.995068,4.832913,13.878868
min,1.000000,6.000000,2012.0,24.000000,21.000000,6.000000,0.000000,37.900000,0.900000,0.100000,1.400000
25%,8.000000,7.000000,2012.0,30.000000,43.250000,14.000000,0.000000,77.650000,7.325000,1.825000,7.400000
50%,16.000000,7.500000,2012.0,34.000000,56.000000,15.000000,0.000000,84.850000,13.150000,4.600000,13.900000
75%,23.000000,8.000000,2012.0,36.000000,66.750000,16.750000,0.475000,89.275000,22.900000,8.625000,23.875000
max,31.000000,9.000000,2012.0,42.000000,90.000000,29.000000,8.700000,96.000000,65.900000,19.000000,68.000000


### 3.2 Duplicate Detection
Before proceeding with analysis, we must ensure there are no identical duplicate rows in the dataset, which could skew our model's training.

In [5]:
df.duplicated().sum()
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")
if duplicate_count > 0:
    df = df.drop_duplicates(keep='first')
    print(f"Duplicates removed. New dataset shape: {df.shape}")
else:
    print("No action required. Dataset is free of duplicates.")

Total duplicate rows found: 0
No action required. Dataset is free of duplicates.


### This describes that data has no dupliactes. Hence the data duplication is handled successfully.

### 3.3 Handling Missing Data

In [6]:
s=df.isnull().sum()
print(f"The null values found in the data are: {s}")

The null values found in the data are: day            0
month          0
year           0
Temperature    0
 RH            0
 Ws            0
Rain           0
FFMC           0
DMC            0
DC             0
ISI            0
BUI            0
FWI            0
Classes        1
dtype: int64


### We check for missing values across all features. Since our missing value is in the Classes column which is our target variable we will drop this row rather than imputing it. Imputing a target variable introduces artificial bias into the model's training phase.

In [7]:
#drop the missing row
df=df.dropna(subset=["Classes  "],axis=0)

### After Droping null value

In [8]:
s=df.isnull().sum()
print(f"The null values found in the data are: {s}")

The null values found in the data are: day            0
month          0
year           0
Temperature    0
 RH            0
 Ws            0
Rain           0
FFMC           0
DMC            0
DC             0
ISI            0
BUI            0
FWI            0
Classes        0
dtype: int64


### Basic illustartion of how our datset has effected in terms of size.

In [9]:
df. info()

<class 'pandas.core.frame.DataFrame'>
Index: 121 entries, 0 to 121
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          121 non-null    int64  
 1   month        121 non-null    int64  
 2   year         121 non-null    int64  
 3   Temperature  121 non-null    int64  
 4    RH          121 non-null    int64  
 5    Ws          121 non-null    int64  
 6   Rain         121 non-null    float64
 7   FFMC         121 non-null    float64
 8   DMC          121 non-null    float64
 9   DC           121 non-null    object 
 10  ISI          121 non-null    float64
 11  BUI          121 non-null    float64
 12  FWI          121 non-null    object 
 13  Classes      121 non-null    object 
dtypes: float64(5), int64(6), object(3)
memory usage: 14.2+ KB


### 3.4 Outlier Detection and Handling

In [10]:
def detect_outliers_iqr(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """
    Detects outliers in specified columns using the IQR method.
    
    Args:
        df (pd.DataFrame): The input dataset.
        features (list): List of numerical column names to check.
        
    Returns:
        pd.DataFrame: A summary table showing the count and percentage of outliers per feature.
    """
    outlier_summary = []
    
    for col in features:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count how many values fall outside the bounds
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = outliers.shape[0]
        outlier_percentage = round((outlier_count / df.shape[0]) * 100, 2)
        
        outlier_summary.append({
            'Feature': col,
            'Outlier Count': outlier_count,
            'Percentage (%)': outlier_percentage
        })
        
    # Convert results to a DataFrame for clean rendering in Jupyter
    return pd.DataFrame(outlier_summary)
# 1. Strip the hidden spaces from all column names permanently
df.columns = df.columns.str.strip()

# 2. Force the string columns into actual numbers (critical for math operations)
df['DC'] = pd.to_numeric(df['DC'], errors='coerce')
df['FWI'] = pd.to_numeric(df['FWI'], errors='coerce')

# 3. Define the clean list (Notice: NO spaces needed in the strings anymore!)
numerical_cols = ['Temperature', 'RH', 'Ws', 'Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']

# 4. Run the outlier detection safely
outlier_report = detect_outliers_iqr(df, numerical_cols)

print("IQR Outlier Detection Summary:")
display(outlier_report)

IQR Outlier Detection Summary:


,Feature,Outlier Count,Percentage (%)
0,Temperature,0,0.00
1,RH,0,0.00
2,Ws,9,7.44
3,Rain,19,15.70
4,FFMC,9,7.44
5,DMC,4,3.31
6,DC,10,8.26
7,ISI,2,1.65
8,BUI,5,4.13
9,FWI,0,0.00


### The outliers are low and droping them might lead to data droping which will effect model training and pattern recognization. Hence the outliers are not treated. 

### 3.5 Handling Inconsistencies (Inspection & Correction)
Before forcing data conversions, we inspect the unique values in our categorical (object) columns. This allows us to identify the exact inconsistencies (like hidden spaces or stray characters) causing pandas to misclassify numerical data as objects.

In [11]:
# 1. THE INSPECTION (Your second point)
# Grab only the columns that pandas thinks are strings/objects
object_columns = df.select_dtypes(include=['object']).columns

print("BEFORE CLEANING: Value Counts for Object Columns\n" + "-"*45)
for col in object_columns:
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("-" * 30)

# 2. THE FORMATTING (Your first point)
# Apply .str.strip() to all object columns at once to remove leading/trailing spaces
df[object_columns] = df[object_columns].apply(lambda x: x.str.strip())

print("\nAFTER STRIPPING: Value Counts for Object Columns\n" + "-"*45)
for col in object_columns:
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("-" * 30)

BEFORE CLEANING: Value Counts for Object Columns
---------------------------------------------
Column: Classes
Classes
fire             78
not fire         41
not fire          1
not fire          1
Name: count, dtype: int64
------------------------------

AFTER STRIPPING: Value Counts for Object Columns
---------------------------------------------
Column: Classes
Classes
fire        78
not fire    43
Name: count, dtype: int64
------------------------------


In [12]:
# 1. Grab only the columns that pandas currently thinks are strings/objects
object_columns = df.select_dtypes(include=['object']).columns

# 2. Strip the hidden spaces from ALL of these text columns
df[object_columns] = df[object_columns].apply(lambda x: x.str.strip())
print("Whitespace stripped from object columns.")

# 3. Target ONLY the specific columns that are supposed to be numbers
columns_to_convert = ['DC', 'FWI']

for col in columns_to_convert:
    # errors='coerce' turns unparseable junk text (like "14.6 9") into NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nData Types After targeted conversion:")
print(df[['DC', 'FWI', 'Classes']].dtypes)

# 4. Final Sanity Check: Did coercing the bad text create any new Nulls?
print("\nNew Null values created by forcing numeric conversion:")
print(df[['DC', 'FWI']].isnull().sum())

Whitespace stripped from object columns.

Data Types After targeted conversion:
DC         float64
FWI        float64
Classes     object
dtype: object

New Null values created by forcing numeric conversion:
DC     0
FWI    0
dtype: int64


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 121 entries, 0 to 121
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          121 non-null    int64  
 1   month        121 non-null    int64  
 2   year         121 non-null    int64  
 3   Temperature  121 non-null    int64  
 4   RH           121 non-null    int64  
 5   Ws           121 non-null    int64  
 6   Rain         121 non-null    float64
 7   FFMC         121 non-null    float64
 8   DMC          121 non-null    float64
 9   DC           121 non-null    float64
 10  ISI          121 non-null    float64
 11  BUI          121 non-null    float64
 12  FWI          121 non-null    float64
 13  Classes      121 non-null    object 
dtypes: float64(7), int64(6), object(1)
memory usage: 14.2+ KB


### Finally the Data Cleaning process is done. The Final data set is free of nans, outliers and inconsistent data formates. 

In [15]:
df.to_csv("Cleaned_Dataset.csv")